# 01. 상태벡터·게이트·측정 기초

이 노트북은 NumPy만으로 1·2 qubit 상태를 만들고 검증한다. QPU나 SDK 없이 수학적 의미를 먼저 확인하는 교육용 실습이다. basis 순서는 `|00>, |01>, |10>, |11>`이고 첫 번째 qubit를 왼쪽 bit로 표기한다.

## 학습 목표

- normalization과 unitary를 자동 검사한다.
- Hadamard와 CNOT으로 Bell state를 만든다.
- Born rule의 정확한 확률과 finite-shot 표본을 구분한다.
- reduced density matrix와 entanglement entropy를 계산한다.

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)
SEED = 20260913
rng = np.random.default_rng(SEED)

def is_unitary(matrix, atol=1e-12):
    identity = np.eye(matrix.shape[0], dtype=complex)
    return np.allclose(matrix.conj().T @ matrix, identity, atol=atol)

def probabilities(state):
    probs = np.abs(state) ** 2
    assert np.isclose(probs.sum(), 1.0), '상태가 정규화되지 않았습니다.'
    return probs

I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
], dtype=complex)

assert all(is_unitary(gate) for gate in (X, H, CNOT))
print('게이트 unitary 검사 통과')

## Bell state 만들기

초기 상태 `|00>`에 첫 qubit의 Hadamard를 적용한 뒤 첫 qubit가 control인 CNOT을 적용한다. 예상 상태는 $(|00>+|11>)/\sqrt{2}$다.

In [ ]:
ket00 = np.array([1, 0, 0, 0], dtype=complex)
after_h = np.kron(H, I) @ ket00
bell = CNOT @ after_h
expected_bell = np.array([1, 0, 0, 1], dtype=complex) / np.sqrt(2)

assert np.allclose(bell, expected_bell)
assert np.isclose(np.vdot(bell, bell).real, 1.0)
print('Bell state amplitude:', bell)
print('정확한 측정 확률:', probabilities(bell))

## 정확한 확률과 finite-shot 결과

표본 오차는 장치 noise가 없어도 생긴다. 한 번의 표본에서 shot 수가 늘수록 오차가 반드시 단조 감소하지는 않으므로, 여러 seed의 평균과 신뢰구간을 사용해야 한다.

In [ ]:
labels = np.array(['00', '01', '10', '11'])
exact_probs = probabilities(bell)

def sample_distribution(state, shots, random_generator):
    counts = dict.fromkeys(labels.tolist(), 0)
    for outcome in random_generator.choice(labels, size=shots, p=probabilities(state)):
        counts[outcome] += 1
    return {label: count / shots for label, count in counts.items()}

for shots in (100, 1_000, 10_000):
    estimate = sample_distribution(bell, shots, rng)
    vector = np.array([estimate[label] for label in labels])
    max_error = np.max(np.abs(vector - exact_probs))
    print(f'{shots:>6} shots | {estimate} | max error={max_error:.4f}')

assert set(label for label, p in sample_distribution(bell, 2_000, rng).items() if p > 0) <= {'00', '11'}

## Reduced density matrix와 얽힘

전체 Bell state는 pure state지만 한 qubit만 보면 최대 혼합 상태다. 계수 행렬 $A$를 사용하면 첫 qubit의 reduced state는 $AA^\dagger$로 계산할 수 있다.

In [ ]:
rho_ab = np.outer(bell, bell.conj())
amplitude_matrix = bell.reshape(2, 2)
rho_a = amplitude_matrix @ amplitude_matrix.conj().T
eigenvalues = np.linalg.eigvalsh(rho_a)
nonzero = eigenvalues[eigenvalues > 1e-15]
entropy_bits = -np.sum(nonzero * np.log2(nonzero))

assert np.allclose(rho_ab @ rho_ab, rho_ab)
assert np.allclose(rho_a, np.eye(2) / 2)
assert np.isclose(entropy_bits, 1.0)
print('첫 qubit의 reduced density matrix:\n', rho_a)
print('entanglement entropy:', entropy_bits, 'bit')

## 직접 해볼 과제

1. 두 번째 qubit에 `Z`를 적용했을 때 amplitude와 측정 확률이 어떻게 달라지는지 설명한다.
2. `(|00>-|11>)/sqrt(2)`가 원래 Bell state와 global phase만 다른지 검사한다.
3. product state `|+0>`의 reduced density matrix와 entropy를 계산한다.
4. shot 수마다 50개 seed를 사용해 max error의 평균과 95% 구간을 구한다.
5. 같은 회로를 Qiskit/PennyLane/Cirq 중 두 SDK로 옮기고 bit order를 명시한다.

완료 조건은 숫자를 얻는 것이 아니라 assertion, convention, seed와 해석을 함께 남기는 것이다.